# Russell3000 All-Stocks Training Notebook

Trains call/put models using all available feature columns from `modelraja.MODELS_DATASET.VW_calls_puts_combined_russell3000`.

In [ ]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "modelraja"
SOURCE_TABLE = "modelraja.MODELS_DATASET.VW_calls_puts_combined_russell3000"

client = bigquery.Client(project=PROJECT_ID)
query = f"SELECT * FROM `{SOURCE_TABLE}`"
df = client.query(query).to_dataframe(create_bqstorage_client=True)
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head(3))
display(pd.DataFrame({"column": df.columns, "dtype": [str(df[c].dtype) for c in df.columns]}))


In [11]:
df.head()

,lookupvalue,nextday_lookupvalue,previousday_lookupvalue,earnings_lookupvalue,fTicker_earnings_date,fTicker_calls_exDate,days_to_earnings,days_earnings_to_expiry,days_to_options_expiry,todays_range,...,calls_diff_strike_current_price,puts_diff_strike_current_price,calls_strike,calls_OpenInterest,puts_openInterest,nextday_fTicker,nextday_Low,nextday_High,nextday_Close_Price,nextday_snapshot_date
0,EMR2024-02-202024-02-23,EMR2024-02-212024-02-23,EMR2024-02-192024-02-23,EMR2024-02-202024-02-072024-02-23,EMR2024-02-07,EMR2024-02-23,-13.0,16.0,3.0,1.59,...,0.20,0.20,105.0,40.0,337.0,EMR,103.86,105.41,104.96,2024-02-21
1,EMR2024-02-202024-03-01,EMR2024-02-212024-03-01,EMR2024-02-192024-03-01,EMR2024-02-202024-02-072024-03-01,EMR2024-02-07,EMR2024-03-01,-13.0,23.0,10.0,1.59,...,0.20,0.20,105.0,24.0,11.0,EMR,103.86,105.41,104.96,2024-02-21
2,CLF2024-05-012024-05-10,CLF2024-05-022024-05-10,CLF2024-04-302024-05-10,CLF2024-05-012024-04-222024-05-10,CLF2024-04-22,CLF2024-05-10,-9.0,18.0,9.0,0.58,...,0.20,0.20,16.5,28.0,45.0,CLF,16.86,17.27,17.26,2024-05-02
3,DHR2024-05-012024-05-10,DHR2024-05-022024-05-10,DHR2024-04-302024-05-10,DHR2024-05-012024-04-232024-05-10,DHR2024-04-23,DHR2024-05-10,-8.0,17.0,9.0,4.97,...,0.61,0.61,247.5,29.0,17.0,DHR,242.66,249.43,246.24,2024-05-02
4,MPC2024-05-012024-05-10,MPC2024-05-022024-05-10,MPC2024-04-302024-05-10,MPC2024-05-012024-04-302024-05-10,MPC2024-04-30,MPC2024-05-10,-1.0,10.0,9.0,7.19,...,1.03,1.03,180.0,31.0,47.0,MPC,179.14,183.81,183.36,2024-05-02


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

TARGET_COL = "price_value_delta_above_call_flag"
ID_COLS = ["Ticker", "Stock_Snapshot_Date"]

# Optional leakage guard. Keep empty to use all columns except targets/IDs.
LEAKAGE_EXCLUDE_COLS = [
    "price_value_delta_above_call_flag",
    "price_value_delta_above_put_flag",
    "nextday_Low",
    "nextday_High",
    "nextday_Close_Price",
    "nextday_snapshot_date",
    "nextday_fTicker",
    "nextday_lookupvalue",
    "price_value_change_nextday_day_high",
    "price_value_change_nextday_day_low",
    "nextday_range",
    "prediction",
    "prob_Beat",
    "prob_NoBeat",
    "actual_outcome",
    "win_loss_flag",
    "selection_score",
    "chosen_action",
    "total_investment",
]


SAFE_FEATURES = [
    "days_to_earnings",
    "days_earnings_to_expiry",
    "days_to_options_expiry",
    "todays_range",
    "strike_to_close_price_gap",
    "Average_True_Range",
    "Relative_Volume",
    "Volume",
    "Average_Volume",
    "Market_Cap",
    "Close_Price",
    "Open",
    "High",
    "Low",
    "Performance_Quarter",
    "Performance_Half_Year",
    "Performance_Week",
    "Performance_Month",
    "Volatility_Week",
    "Volatility_Month",
    "Target_Price",
    "Beta",
    "calls_lastClose",
    "puts_lastClose",
    "calls_strike",
    "calls_OpenInterest",
    "puts_openInterest",
    "calls_diff_strike_current_price",
    "puts_diff_strike_current_price",
]

LEAKAGE_EXCLUDE_COLS = [
    "price_value_delta_above_call_flag",
    "price_value_delta_above_put_flag",
    "nextday_Low",
    "nextday_High",
    "nextday_Close_Price",
    "nextday_snapshot_date",
    "nextday_fTicker",
    "nextday_lookupvalue",
    "price_value_change_nextday_day_high",
    "price_value_change_nextday_day_low",
    "nextday_range",
    "prediction",
    "prob_Beat",
    "prob_NoBeat",
    "actual_outcome",
    "win_loss_flag",
    "selection_score",
    "chosen_action",
    "total_investment",
]

working_df = df.copy()
# Convert percentage-like columns to numeric fraction (e.g., "7.5%" -> 0.075)
percent_cols = ["Performance_Week", "Performance_Half_Year", "Volatility_Week", "Change"]
for col in percent_cols:
    if col in working_df.columns:
        working_df[col] = (
            working_df[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        working_df[col] = pd.to_numeric(working_df[col], errors="coerce") / 100.0

print("Configured target:", TARGET_COL)


# Accelerator config
USE_GPU = True  # set True to attempt GPU via XGBoost device="cuda"
APPLY_SKLEARN_INTEL_PATCH = True  # faster CPU path if sklearn-intelex is installed





In [ ]:

# Optional sklearn-intelex patch (CPU acceleration)
if APPLY_SKLEARN_INTEL_PATCH:
    try:
        from sklearnex import patch_sklearn
        patch_sklearn()
        print("Applied sklearn-intelex patch for CPU acceleration.")
    except Exception:
        print("sklearn-intelex not available; continuing with standard sklearn.")

def make_deterministic_split(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    ticker = out.get("Ticker", pd.Series(index=out.index, dtype="object")).fillna("").astype(str)
    snap = pd.to_datetime(out.get("Stock_Snapshot_Date", pd.Series(index=out.index)), errors="coerce").astype(str)
    key = ticker + "|" + snap
    bucket = (pd.util.hash_pandas_object(key, index=False).astype("uint64") % 100).astype(int)

    out["split"] = "test"
    out.loc[bucket < 70, "split"] = "train"
    out.loc[(bucket >= 70) & (bucket < 85), "split"] = "validation"
    return out

def train_one_target(input_df: pd.DataFrame, target_col: str, model_tag: str):
    print(f"\n===== Training {model_tag} target: {target_col} =====")

    if target_col not in input_df.columns:
        raise ValueError(f"Missing target column: {target_col}")

    train_df = input_df[input_df[target_col].notna()].copy()
    if train_df.empty:
        raise ValueError(f"No non-null rows for target: {target_col}")

    train_df = make_deterministic_split(train_df)
    print("Split counts:")
    print(train_df["split"].value_counts(dropna=False))

    drop_cols = ["price_value_delta_above_call_flag", "price_value_delta_above_put_flag", "split"] + ID_COLS + LEAKAGE_EXCLUDE_COLS
    base_cols = [c for c in train_df.columns if c not in drop_cols]
    feature_cols = [c for c in SAFE_FEATURES if c in base_cols]
    if not feature_cols:
        raise ValueError("No SAFE_FEATURES found in input data after exclusions.")
    print("Using", len(feature_cols), "safe features")
    print(feature_cols)

    X_train = train_df.loc[train_df["split"] == "train", feature_cols].copy()
    y_train = train_df.loc[train_df["split"] == "train", target_col].copy()
    X_val = train_df.loc[train_df["split"] == "validation", feature_cols].copy()
    y_val = train_df.loc[train_df["split"] == "validation", target_col].copy()
    X_test = train_df.loc[train_df["split"] == "test", feature_cols].copy()
    y_test = train_df.loc[train_df["split"] == "test", target_col].copy()

    if X_train.empty or X_val.empty or X_test.empty:
        raise ValueError("One or more splits are empty; check source data.")

    numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), numeric_features),
            ("cat", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), categorical_features),
        ]
    )

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_val_enc = le.transform(y_val)
    y_test_enc = le.transform(y_test)

    xgb_params = dict(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
    )

    if USE_GPU:
        xgb_params["device"] = "cuda"

    model_xgb = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(**xgb_params)),
    ])

    try:
        model_xgb.fit(X_train, y_train_enc)
        if USE_GPU:
            print("XGBoost trained with GPU (device=cuda).")
    except Exception as e:
        if USE_GPU:
            print(f"GPU training failed ({e}); falling back to CPU.")
            xgb_params.pop("device", None)
            model_xgb = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("classifier", XGBClassifier(**xgb_params)),
            ])
            model_xgb.fit(X_train, y_train_enc)
        else:
            raise
    pred_val_base = le.inverse_transform(model_xgb.predict(X_val))
    print("Base Validation Accuracy:", accuracy_score(y_val, pred_val_base))

    param_dist = {
        "classifier__n_estimators": [100, 200, 300, 500],
        "classifier__max_depth": [3, 5, 6, 8],
        "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "classifier__subsample": [0.6, 0.8, 1.0],
        "classifier__colsample_bytree": [0.6, 0.8, 1.0],
        "classifier__gamma": [0, 1, 5],
    }

    search = RandomizedSearchCV(
        estimator=model_xgb,
        param_distributions=param_dist,
        n_iter=20,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X_train, y_train_enc)
    best_model = search.best_estimator_

    pred_val = le.inverse_transform(best_model.predict(X_val))
    pred_test = le.inverse_transform(best_model.predict(X_test))

    print("Best params:", search.best_params_)
    print("Best CV score:", search.best_score_)
    print("Tuned Validation Accuracy:", accuracy_score(y_val, pred_val))
    print("Tuned Test Accuracy:", accuracy_score(y_test, pred_test))
    print("Tuned Test Classification Report:\n", classification_report(y_test, pred_test))

    perm = permutation_importance(
        best_model,
        X_val,
        y_val_enc,
        n_repeats=8,
        random_state=42,
        scoring="accuracy",
        n_jobs=-1,
    )
    imp_df = pd.DataFrame({
        "feature": X_val.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }).sort_values("importance_mean", ascending=False)

    print(f"Top 30 significant features for {model_tag}:")
    display(imp_df.head(30))

    test_out = train_df.loc[train_df["split"] == "test", ID_COLS].copy() if all(c in train_df.columns for c in ID_COLS) else pd.DataFrame(index=X_test.index)
    test_out["target_type"] = model_tag
    test_out["actual"] = y_test.values
    test_out["predicted"] = pred_test
    if hasattr(best_model.named_steps["classifier"], "predict_proba"):
        probs = best_model.predict_proba(X_test)
        for i, cls in enumerate(le.classes_):
            test_out[f"prob_{cls}"] = probs[:, i]

    return {
        "best_model": best_model,
        "model_xgb": model_xgb,
        "label_encoder": le,
        "importance_df": imp_df,
        "test_predictions": test_out,
    }





In [ ]:
artifacts = {}
if TARGET_COL in working_df.columns:
    artifacts['call'] = train_one_target(working_df, TARGET_COL, 'call')
else:
    print(f"Skipping call: missing target {TARGET_COL}")

print('Trained target:', list(artifacts.keys()))



In [ ]:
import joblib
from pathlib import Path

out_dir = Path('models')
out_dir.mkdir(exist_ok=True)

saved = []
if 'call' in artifacts:
    joblib.dump(artifacts['call']['best_model'], out_dir / 'best_model_call_r3000.joblib')
    joblib.dump(artifacts['call']['model_xgb'], out_dir / 'model_xgb_call_r3000.joblib')
    joblib.dump(artifacts['call']['label_encoder'], out_dir / 'label_encoder_call_r3000.joblib')
    artifacts['call']['test_predictions'].to_csv('xgboost_tuned_predictions_call_r3000.csv', index=False)
    artifacts['call']['importance_df'].to_csv('feature_importance_call_r3000.csv', index=False)
    saved += ['best_model_call_r3000.joblib', 'model_xgb_call_r3000.joblib', 'label_encoder_call_r3000.joblib', 'xgboost_tuned_predictions_call_r3000.csv', 'feature_importance_call_r3000.csv']

print('Saved artifacts:')
for s in saved:
    print('-', s)



In [ ]:
# Feature-set experiment runner (quick comparison loop)
# Uses train_one_target(...) already defined above.

FEATURE_EXPERIMENTS = {
    "safe_all": SAFE_FEATURES,
    "core_price_volume": [
        "Stock_Price", "Close_Price", "Open", "High", "Low",
        "Volume", "Average_Volume", "Relative_Volume",
        "Performance_Week", "Performance_Month", "Performance_Quarter", "Performance_Half_Year",
        "Volatility_Week", "Average_True_Range", "days_to_earnings",
    ],
    "options_focused": [
        "calls_lastClose", "puts_lastClose", "calls_strike",
        "calls_OpenInterest", "puts_openInterest",
        "calls_diff_strike_current_price", "puts_diff_strike_current_price",
        "strike_to_close_price_gap", "days_to_options_expiry", "days_earnings_to_expiry",
    ],
    "macro_plus_options": [
        "Target_Price", "Beta", "Market_Cap",
        "Performance_Week", "Performance_Month", "Performance_Quarter", "Performance_Half_Year",
        "Volatility_Week", "Average_True_Range",
        "calls_lastClose", "puts_lastClose", "calls_OpenInterest", "puts_openInterest",
        "strike_to_close_price_gap", "days_to_earnings",
    ],
}

if "TARGET_COL" not in globals():
    raise ValueError("TARGET_COL not found. Run prior config/training cells first.")

exp_rows = []
importance_frames = []
orig_safe = list(SAFE_FEATURES)

for exp_name, exp_features in FEATURE_EXPERIMENTS.items():
    print(f"\n===== Running experiment: {exp_name} =====")
    SAFE_FEATURES = [f for f in exp_features if f in working_df.columns]
    if not SAFE_FEATURES:
        print("No valid features found in dataset for this experiment. Skipping.")
        continue

    art = train_one_target(working_df, TARGET_COL, f"{TARGET_COL}_{exp_name}")

    # Extract compact metrics from the test predictions
    test_df = art["test_predictions"].copy()
    acc = (test_df["actual"].astype(str) == test_df["predicted"].astype(str)).mean()

    imp = art["importance_df"].copy()
    imp["experiment"] = exp_name
    imp["importance_rank"] = imp["importance_mean"].rank(ascending=False, method="dense")
    importance_frames.append(imp)

    top_feats = imp.sort_values("importance_mean", ascending=False).head(10)["feature"].tolist()

    exp_rows.append({
        "experiment": exp_name,
        "n_features": len(SAFE_FEATURES),
        "test_accuracy": float(acc),
        "top_10_features": ", ".join(top_feats),
    })

# restore original safe list
SAFE_FEATURES = orig_safe

experiment_summary = pd.DataFrame(exp_rows).sort_values("test_accuracy", ascending=False)
print("\nExperiment summary:")
display(experiment_summary)

# Per-experiment ranked feature importances
if importance_frames:
    all_importance = pd.concat(importance_frames, ignore_index=True)
    print("\nFeature importance ranking by experiment:")
    display(all_importance.sort_values(["experiment", "importance_mean"], ascending=[True, False]))

    # Cross-experiment ranking: higher mean importance and better average rank = more significant.
    feature_rank_summary = (
        all_importance.groupby("feature", as_index=False)
        .agg(
            mean_importance=("importance_mean", "mean"),
            std_importance=("importance_mean", "std"),
            mean_rank=("importance_rank", "mean"),
            seen_in_experiments=("experiment", "nunique"),
        )
        .sort_values(["mean_importance", "seen_in_experiments", "mean_rank"], ascending=[False, False, True])
    )

    print("\nOverall ranked feature importance (across experiments):")
    display(feature_rank_summary)

    all_importance.to_csv(f"feature_importance_all_experiments_{TARGET_COL}.csv", index=False)
    feature_rank_summary.to_csv(f"feature_importance_ranked_summary_{TARGET_COL}.csv", index=False)
    print(f"Saved: feature_importance_all_experiments_{TARGET_COL}.csv")
    print(f"Saved: feature_importance_ranked_summary_{TARGET_COL}.csv")

# Optional export
experiment_summary.to_csv(f"feature_experiment_summary_{TARGET_COL}.csv", index=False)
print(f"Saved: feature_experiment_summary_{TARGET_COL}.csv")

